# Notebook 07: CI/CD and Deployment

Welcome to the seventh notebook in our MLOps workshop. So far you have built data pipelines, tracked experiments, trained models, served predictions via an API, and monitored for drift. But how does all of this get from your laptop into production **safely and automatically**?

That is the job of **CI/CD** -- Continuous Integration and Continuous Deployment.

### What You Will Learn

- What CI/CD means and why it matters for ML systems
- How to read and understand a GitHub Actions CI pipeline
- The three levels of testing in ML (unit, integration, model validation)
- Docker: packaging your application so it runs anywhere
- Docker Compose: running multiple services together
- Kubernetes: deploying at scale in production

---
## 1. What is CI/CD?

**CI** stands for **Continuous Integration**. Every time someone pushes code to the repository, an automated system runs a suite of checks: linting, type checking, unit tests, integration tests. If anything fails, the team is notified immediately. The idea is simple -- catch bugs early, before they reach production.

**CD** stands for **Continuous Deployment** (or Continuous Delivery). If all CI checks pass, the system automatically deploys the new version. No human needs to manually copy files to a server, restart services, or run scripts.

### Why CI/CD is Different for ML

In traditional software, CI/CD tests **code**. In ML systems, you also need to test:

- **Data quality** -- Has the input data changed format? Are there new missing values?
- **Model quality** -- Is the new model actually better than the current one?
- **Prediction quality** -- Do the predictions make sense? Are they within expected ranges?

A CI pipeline for an ML system does not just check that the code runs. It checks that the **model is good enough to deploy**.

```
Traditional CI/CD:              ML CI/CD:

Push code                       Push code / new data / new model
   |                               |
   v                               v
Run tests                       Run code tests
   |                               |
   v                               v
Deploy                          Run data validation tests
                                   |
                                   v
                                Run model quality tests
                                   |
                                   v
                                Deploy (if all pass)
```

Let's look at how our project implements this.

---
## 2. Our CI Pipeline

We use **GitHub Actions** for CI. GitHub Actions is a free CI/CD platform built into GitHub. Every time code is pushed, it spins up a virtual machine, installs dependencies, and runs our checks.

The pipeline is defined in a YAML file at `.github/workflows/ci.yml`. Let's read it.

In [ ]:
# Read and display our CI configuration
with open("../.github/workflows/ci.yml") as f:
    ci_config = f.read()

print(ci_config)

### Understanding Each Step

Let's break down what this CI pipeline does:

**Trigger** (`on`): The pipeline runs on every push to `main` or `develop`, and on every pull request targeting `main`. This means every code change is tested before it can be merged.

**Concurrency**: If you push twice quickly, the first run is cancelled. No wasted compute.

**Job 1 -- Lint** (`lint`):
- Checks out the code (`actions/checkout@v4`)
- Sets up Python 3.11 (`actions/setup-python@v5`)
- Installs Ruff, a fast Python linter
- Runs `ruff check` -- catches style issues, unused imports, potential bugs
- Runs `ruff format --check` -- verifies code is consistently formatted

**Job 2 -- Type Check** (`type-check`):
- Installs the full project (`pip install -e ".[dev]"`)
- Runs `mypy` -- a static type checker that catches type errors before runtime
- Example: if a function expects a `float` but you pass a `str`, mypy catches it

**Job 3 -- Test** (`test`):
- Depends on `lint` passing first (`needs: [lint]`)
- Installs test dependencies
- Runs `pytest` on all unit tests with coverage tracking
- Uploads coverage report to Codecov

The key insight: **lint and type-check run in parallel** (they have no `needs`), but **tests only run after linting passes**. No point running slow tests if the code does not even pass basic quality checks.

---
## 3. Testing in ML

Testing in ML systems happens at three levels, like layers of defense:

### Level 1: Unit Tests
Test individual functions in isolation. Does `compute_mae()` return the correct value for known inputs? Does `create_lag_features()` produce the right number of columns?

### Level 2: Integration Tests
Test that components work together. Does the feature engineering pipeline produce a DataFrame that the model can consume? Does the API return valid JSON when given a prediction request?

### Level 3: Model Validation Tests
Test model quality against thresholds. Is the RMSE below 50 kWh? Is the model better than a simple baseline? Does it perform consistently across different building types?

```
+-------------------------------------------+
|      Model Validation Tests               |  <-- Slow, run less often
|  (RMSE < threshold, better than baseline) |
+-------------------------------------------+
|      Integration Tests                    |
|  (pipeline end-to-end, API contracts)     |
+-------------------------------------------+
|      Unit Tests                           |  <-- Fast, run every push
|  (individual functions, edge cases)       |
+-------------------------------------------+
```

Let's run some of our actual unit tests to see this in action.

In [ ]:
# Run our metrics unit tests -- these test individual evaluation functions
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/unit/test_metrics.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd="/home/user/Energy-Demand-Forecasting-MLOps-System",
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Run data validator tests -- these test data quality checking logic
result = subprocess.run(
    ["python", "-m", "pytest", "tests/unit/test_validator.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd="/home/user/Energy-Demand-Forecasting-MLOps-System",
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Run feature engineering tests
result = subprocess.run(
    ["python", "-m", "pytest", "tests/unit/test_feature_engineering.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd="/home/user/Energy-Demand-Forecasting-MLOps-System",
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

Notice how fast unit tests run. Each test checks one specific behavior:
- Does MAE compute correctly for known inputs?
- Does the validator reject data with missing columns?
- Do lag features have the right number of NaN rows?

These run on every single push. If someone accidentally breaks a metric calculation, the CI pipeline catches it in seconds -- not after a bad model gets deployed.

---
## 4. Docker: "It Works on My Machine" is Not Good Enough

### The Problem

You trained your model on your laptop with Python 3.11, NumPy 1.24, and XGBoost 1.7. The production server has Python 3.9, NumPy 1.21, and XGBoost 2.0. Your model crashes. Or worse, it runs but gives different predictions because of a subtle library difference.

This is the **"works on my machine"** problem, and it has plagued software engineering for decades.

### The Solution: Docker

Docker packages your application along with **everything it needs to run** -- the operating system, Python version, all libraries, your code, and your configuration files -- into a single **container**.

Think of a Docker container like a shipping container. It does not matter what ship (server) carries it -- the contents are always the same. A Docker container that works on your laptop will work identically on any server, any cloud, anywhere.

```
Without Docker:                    With Docker:

+----------+  +----------+        +-------------------+
| Your App |  | Your App |        | +---------------+ |
+----------+  +----------+        | | Your App      | |
| Python   |  | Python   |        | | Python 3.11   | |
| 3.11     |  | 3.9      |        | | NumPy 1.24    | |
| NumPy    |  | NumPy    |        | | XGBoost 1.7   | |
| 1.24     |  | 1.21     |        | | Ubuntu 22.04  | |
+----------+  +----------+        | +---------------+ |
| Your     |  | Prod     |        | Docker Container  |
| Laptop   |  | Server   |        +-------------------+
+----------+  +----------+        Runs IDENTICALLY
  DIFFERENT!    DIFFERENT!        on any machine.
```

### Key Docker Concepts

- **Image**: A blueprint for a container. Like a class in Python.
- **Container**: A running instance of an image. Like an object.
- **Dockerfile**: A recipe that describes how to build an image.
- **Registry**: A place to store and share images (like Docker Hub or GitHub Container Registry).

Let's look at our Dockerfile.

In [ ]:
# Read and display our Dockerfile
with open("../Dockerfile") as f:
    dockerfile = f.read()

print(dockerfile)

### Understanding the Dockerfile Line by Line

This is a **multi-stage build** -- it uses two separate stages to keep the final image small.

**Stage 1 -- Builder** (`FROM python:3.11-slim as builder`):
- Starts from a slim Python 3.11 base image
- Copies in `pyproject.toml` and `src/` -- just what is needed to build
- Runs `pip install build` and `python -m build --wheel` to create a wheel file
- This stage is thrown away after building. It is only used to compile the package.

**Stage 2 -- Runtime** (`FROM python:3.11-slim`):
- Starts fresh from a clean Python image (no build tools)
- Installs only `curl` (needed for health checks)
- Creates a non-root user `appuser` for security
- Copies the wheel from Stage 1 and installs it
- Copies configuration files
- Sets the non-root user
- Exposes port 8000
- Adds a health check (Docker will restart the container if `/health` fails)
- Sets the startup command: run uvicorn to serve the FastAPI app

**Why multi-stage?** The builder stage has compilers, build tools, and source code. The runtime stage has only what is needed to run. This makes the final image smaller and more secure.

**Common Docker commands:**
```bash
# Build the image
docker build -t energy-forecast:latest .

# Run a container from the image
docker run -p 8000:8000 energy-forecast:latest

# List running containers
docker ps

# Stop a container
docker stop <container-id>
```

---
## 5. Docker Compose: Running Multiple Services Together

Our system is not just one container. It is many:
- **PostgreSQL** -- database for MLflow and Airflow
- **MinIO** -- S3-compatible storage for model artifacts
- **MLflow** -- experiment tracking and model registry
- **API** -- our FastAPI prediction service
- **Airflow** -- orchestration (webserver + scheduler)
- **Prometheus** -- metrics collection
- **Grafana** -- dashboards

Running each one manually with `docker run` would be tedious and error-prone. **Docker Compose** lets you define all services in a single YAML file and start them with one command.

```bash
# Start everything
docker compose up -d

# Stop everything
docker compose down
```

Let's look at our `docker-compose.yml`.

In [ ]:
# Read and display the first 60 lines of docker-compose.yml
with open("../docker-compose.yml") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"\nFirst 60 lines:\n")
print("".join(lines[:60]))

### Key Compose Concepts

**Services**: Each block under `services:` defines one container. We have `postgres`, `minio`, `mlflow`, `api`, `airflow-webserver`, `airflow-scheduler`, `prometheus`, and `grafana`.

**Shared Configuration** (`x-airflow-common`): The `&airflow-common` anchor defines settings shared by all Airflow containers. This avoids duplication -- the webserver and scheduler both use `<<: *airflow-common`.

**Environment Variables**: Each service gets its configuration through environment variables. Secrets come from a `.env` file (`env_file: .env`).

**Volumes**: Persistent data is stored in named volumes (`postgres-data`, `minio-data`). These survive container restarts.

**Health Checks**: Each service defines how Docker should check if it is healthy. For example, PostgreSQL uses `pg_isready`, and the API uses `curl -f http://localhost:8000/health`.

**Dependencies** (`depends_on`): Services start in the right order. The API waits for MLflow to be healthy. MLflow waits for PostgreSQL. Airflow waits for both.

**Networks**: All services are on the `energy-net` network so they can communicate by service name (e.g., the API can reach MLflow at `http://mlflow:5000`).

**Ports**: Services expose ports to the host machine:
- API: `localhost:8000`
- MLflow: `localhost:5000`
- Airflow: `localhost:8080`
- Grafana: `localhost:3000`
- Prometheus: `localhost:9090`

---
## 6. Kubernetes: Deploying at Scale

### From Docker to Kubernetes

Docker Compose is great for development and small deployments. But what happens when your API needs to handle thousands of requests per second? Or when a container crashes at 3 AM and nobody is around to restart it?

**Kubernetes (K8s)** is a container orchestration platform. If Docker runs **one container**, Kubernetes runs **thousands** -- across many machines, with automatic scaling, self-healing, and rolling updates.

```
Docker:                          Kubernetes:
+----------+                     +---------------------------------+
|Container |                     | +----+ +----+ +----+ +----+    |
|          |                     | |Pod | |Pod | |Pod | |Pod |    |
+----------+                     | +----+ +----+ +----+ +----+    |
1 container,                     | +----+ +----+ +----+ +----+    |
1 machine                        | |Pod | |Pod | |Pod | |Pod |    |
                                 | +----+ +----+ +----+ +----+    |
                                 | Node 1       | Node 2          |
                                 +---------------------------------+
                                 Thousands of containers,
                                 many machines, auto-scaling
```

### Key Kubernetes Concepts

- **Pod**: The smallest unit. A Pod runs one (or a few related) containers. Think of it as a wrapper around your Docker container.
- **Deployment**: Manages a set of identical Pods. You say "I want 3 replicas" and Kubernetes ensures exactly 3 are always running.
- **Service**: A stable network endpoint for a set of Pods. Pods come and go, but the Service address stays the same.
- **HPA (Horizontal Pod Autoscaler)**: Automatically scales the number of Pods based on CPU/memory usage. High traffic? More Pods. Low traffic? Fewer Pods.
- **ConfigMap**: Stores configuration as key-value pairs, injected into Pods as environment variables.
- **Namespace**: A way to organize resources. Like a folder for your Kubernetes objects.

Let's look at our Kubernetes manifests.

In [ ]:
import os

k8s_dir = "../deployment/kubernetes"

# Read and display all K8s manifests
manifest_order = ["namespace.yaml", "configmap.yaml", "deployment.yaml", "service.yaml", "hpa.yaml"]

for filename in manifest_order:
    filepath = os.path.join(k8s_dir, filename)
    if os.path.exists(filepath):
        print("=" * 60)
        print(f"FILE: {filename}")
        print("=" * 60)
        with open(filepath) as f:
            print(f.read())
        print()

### Understanding the Manifests

**namespace.yaml**: Creates an isolated namespace `energy-forecast`. All our resources live here, separate from other applications on the cluster.

**configmap.yaml**: Stores configuration that the API needs -- MLflow URI, log level, API port, CORS settings, etc. These become environment variables in the container.

**deployment.yaml**: This is the heart of the deployment.
- `replicas: 2` -- always keep 2 copies running for high availability
- `strategy: RollingUpdate` -- when deploying a new version, start new Pods before stopping old ones (zero downtime)
- `maxSurge: 1, maxUnavailable: 0` -- never have fewer than 2 Pods during an update
- `resources` -- CPU and memory requests/limits prevent one container from starving others
- `livenessProbe` -- Kubernetes restarts the Pod if `/health` fails (self-healing)
- `readinessProbe` -- Kubernetes only sends traffic to Pods that are ready
- `securityContext` -- runs as non-root with a read-only filesystem (defense in depth)

**service.yaml**: Creates a stable network address `energy-forecast-api.energy-forecast.svc.cluster.local`. Load balances traffic across all healthy Pods.

**hpa.yaml**: The Horizontal Pod Autoscaler.
- Min 2 replicas, max 10
- Scales up when CPU > 70% or memory > 80%
- Scale-up is aggressive (2 Pods at a time, every 60s)
- Scale-down is conservative (1 Pod at a time, every 120s, with 5-minute stabilization)

```bash
# Deploy to Kubernetes
kubectl apply -f deployment/kubernetes/

# Check the status
kubectl get pods -n energy-forecast

# Watch auto-scaling in action
kubectl get hpa -n energy-forecast --watch
```

---
## 7. Exercises

### Exercise 1: Add a Model Validation Step to CI

Our current CI pipeline checks code quality (lint, type-check, test). But it does not validate model quality. Design a new job in the CI pipeline that would:

1. Train a model on a small sample of data
2. Evaluate it against known thresholds (e.g., RMSE < 100)
3. Fail the pipeline if the model does not meet the threshold

Write the YAML for this job. Where in the `needs` chain should it go?

### Exercise 2: Optimize the Dockerfile

Our Dockerfile copies `pyproject.toml` and `src/` into the builder. This means that changing a single line of code rebuilds all dependencies. Using Docker layer caching, how would you restructure the Dockerfile so that dependencies are only reinstalled when `pyproject.toml` changes?

Hint: Copy `pyproject.toml` first, install dependencies, then copy `src/`.

---
## 8. Key Takeaways

1. **CI/CD automates quality gates.** Every code push is tested automatically. If tests fail, the code does not get deployed. This catches bugs before they reach users.

2. **ML CI/CD goes beyond code tests.** You also need data validation tests and model quality tests. A model that produces bad predictions is worse than no model at all.

3. **Docker solves the environment problem.** By packaging your app with all its dependencies, you guarantee it runs the same everywhere -- on your laptop, in CI, and in production.

4. **Docker Compose orchestrates multiple services locally.** One command starts your entire stack: database, model registry, API, monitoring, and orchestration.

5. **Kubernetes takes you to production scale.** Auto-scaling, self-healing, rolling updates, and resource management let you serve thousands of requests reliably.

6. **Testing happens at three levels.** Unit tests (fast, every push) catch function-level bugs. Integration tests catch component interaction issues. Model validation tests catch quality regressions.

In the next notebook, we will look at **orchestration** -- how Airflow coordinates all these pipelines into a cohesive system.